# 10. Final project: an end-to-end image-analysis workflow

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner  
**Estimated study time:** 30–60 minutes

## What you will learn
- Combine the full workflow
- Validate segmentation visually
- Produce a quantitative measurement table

> **Course habit:** run one cell at a time, inspect the result, and change one parameter before moving on.

## Final project

Combine the core skills without adding a new library: inspect → preprocess → segment → validate → measure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skimage as ski

def show_image(image, title='', cmap=None, figsize=(6, 5)):
    """Display one image with a clean, repeatable layout."""
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
from skimage import filters, morphology, measure, color
image=ski.data.cell()
print('shape:',image.shape)
print('dtype:',image.dtype)
print('range:',image.min(),'to',image.max())
show_image(image,'Input cell image',cmap='gray')

## Step 1 — smooth just enough to reduce fine noise

In [ ]:
smooth=filters.gaussian(image,sigma=1.2)
show_image(smooth,'Gaussian-smoothed',cmap='gray')

## Step 2 — segment bright structures

In [ ]:
threshold=filters.threshold_otsu(smooth)
mask=smooth>threshold
mask=morphology.remove_small_objects(mask,max_size=39)
mask=morphology.remove_small_holes(mask,max_size=39)
show_image(mask,'Cleaned binary mask',cmap='gray')

## Step 3 — label and inspect

In [ ]:
labels=measure.label(mask)
overlay=color.label2rgb(labels,image=image,bg_label=0,alpha=0.35)
show_image(overlay,f'Segmentation overlay: {labels.max()} labeled regions')

## Step 4 — measure

In [ ]:
props=measure.regionprops_table(labels,intensity_image=image,properties=('label','area','centroid','mean_intensity','eccentricity','solidity'))
results=pd.DataFrame(props)
print(results.head())

## Step 5 — ask whether the result is believable

In [ ]:
print('number of objects:',len(results))
print('median area:',results['area'].median())
print('median mean intensity:',results['mean_intensity'].median())
print(results.describe())

## Final checks
1. Does the mask follow the structures you care about?
2. Are obvious objects missing?
3. Are objects incorrectly merged or split?
4. Are measurements in pixels or calibrated physical units?
5. Would the settings work on darker/noisier images?

## Optional extension
Replace `ski.data.cell()` with your own image and change one parameter at a time.